# Q-Learning — Análise de Resultados

Este notebook carrega o checkpoint de um treino Q-Learning no GridWorld 10×10
e gera visualizações para análise do aprendizado:

1. **Curva de Aprendizado** — retorno por episódio (rolling average)
2. **Mapa de Calor dos Valores** — V(s) = max_a Q(s,a)
3. **Política Aprendida** — setas indicando a ação greedy em cada célula
4. **Caminho Greedy** — trajetória do agente treinado
5. **Métricas de Convergência** — taxa de sucesso, comprimento dos episódios

In [ ]:
import os
import sys
from pathlib import Path

# Garantir que o working directory é a raiz do projeto
_nb_dir = Path.cwd()
if _nb_dir.name == "notebooks":
    os.chdir(_nb_dir.parent)

PROJECT_ROOT = Path.cwd()
print(f"Working directory: {PROJECT_ROOT}")

# Garantir que src/ está no path (funciona mesmo sem pip install -e)
SRC_DIR = str(PROJECT_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from rl_tabular.envs import GridWorldEnv
from rl_tabular.agents import QLearningAgent
from rl_tabular.utils.visualization import (
    plot_value_heatmap,
    plot_policy_arrows,
    plot_value_and_policy,
    plot_greedy_path,
    plot_learning_curve,
    greedy_rollout,
)

# Estilo dos gráficos
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

## 1. Carregar Configuração e Reconstruir Ambiente + Agente

In [ ]:
RUN_DIR = "runs/sarsa_20x20"

# Carregar a config usada no treino
with open(f"{RUN_DIR}/config.yaml") as f:
    config = yaml.safe_load(f)

env_cfg = config["environment"]
agent_cfg = config["agent"]

print(f"Run: {config['run_name']}")
print(f"Grid: {env_cfg['width']}×{env_cfg['height']}")
print(f"Obstáculos: mode={env_cfg['obstacle_mode']}, density={env_cfg['obstacle_density']}")
print(f"Agente: α={agent_cfg['alpha']}, γ={agent_cfg['gamma']}")

In [ ]:
# Reconstruir o ambiente com a mesma seed (garante o mesmo grid)
env = GridWorldEnv(
    width=env_cfg["width"],
    height=env_cfg["height"],
    obstacle_density=env_cfg["obstacle_density"],
    obstacle_mode=env_cfg["obstacle_mode"],
    seed=env_cfg["seed"],
    cluster_size=env_cfg.get("cluster_size", 5),
    allow_diagonal=env_cfg.get("allow_diagonal", False),
    reward_goal=env_cfg.get("reward_goal", 100.0),
    reward_obstacle=env_cfg.get("reward_obstacle", -10.0),
    reward_step=env_cfg.get("reward_step", -0.01),
    shaping=env_cfg.get("shaping"),
    max_steps=env_cfg.get("max_steps", 500),
    render_mode="ansi",
)

print(f"Obstáculos no grid: {env.grid.sum()}/{env.width * env.height}")
print(f"Start: {env.start}, Goal: {env.goal}")
print(f"Ações: {env.action_space.n} ({'8-dir' if env.allow_diagonal else '4-dir'})")
print()
print("Grid (ASCII):")
env.reset()
print(env.render())

In [ ]:
# Reconstruir o agente e carregar o checkpoint
agent = QLearningAgent(
    num_states=env.observation_space.n,
    num_actions=env.action_space.n,
    alpha=agent_cfg["alpha"],
    gamma=agent_cfg["gamma"],
)
agent.load(f"{RUN_DIR}/checkpoints/latest")

print(f"Q-table carregada: shape={agent.Q.shape}")
print(f"Epsilon final: {agent.epsilon}")
print(f"Entradas não-zero na Q-table: {np.count_nonzero(agent.Q)}/{agent.Q.size}")

## 2. Curva de Aprendizado

In [ ]:
df = pd.read_csv(f"{RUN_DIR}/train_history.csv")

print(f"Total de episódios: {len(df)}")
print(f"Taxa de sucesso global: {df['success'].mean():.2%}")
print(f"Retorno médio (últimos 1000 ep): {df['episode_return'].tail(1000).mean():.2f}")
print(f"Comprimento médio (últimos 1000 ep): {df['episode_length'].tail(1000).mean():.1f} passos")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Retorno (rolling avg) ---
window = 500
rolling_return = df["episode_return"].rolling(window, min_periods=1).mean()
axes[0, 0].plot(df["episode"], rolling_return, linewidth=0.8, color="#2196F3")
axes[0, 0].set_xlabel("Episódio")
axes[0, 0].set_ylabel(f"Retorno (média móvel, w={window})")
axes[0, 0].set_title("Curva de Aprendizado — Retorno")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=100, color="green", linestyle="--", alpha=0.5, label="Retorno máximo teórico")
axes[0, 0].legend()

# --- Comprimento do episódio (rolling avg) ---
rolling_length = df["episode_length"].rolling(window, min_periods=1).mean()
axes[0, 1].plot(df["episode"], rolling_length, linewidth=0.8, color="#FF9800")
axes[0, 1].set_xlabel("Episódio")
axes[0, 1].set_ylabel(f"Passos por episódio (média móvel, w={window})")
axes[0, 1].set_title("Comprimento dos Episódios")
axes[0, 1].grid(True, alpha=0.3)

# --- Taxa de sucesso (rolling avg) ---
rolling_success = df["success"].rolling(window, min_periods=1).mean()
axes[1, 0].plot(df["episode"], rolling_success * 100, linewidth=0.8, color="#4CAF50")
axes[1, 0].set_xlabel("Episódio")
axes[1, 0].set_ylabel(f"Taxa de sucesso % (média móvel, w={window})")
axes[1, 0].set_title("Taxa de Sucesso")
axes[1, 0].set_ylim(-5, 105)
axes[1, 0].grid(True, alpha=0.3)

# --- Epsilon ao longo do treino ---
axes[1, 1].plot(df["episode"], df["epsilon"], linewidth=0.8, color="#9C27B0")
axes[1, 1].set_xlabel("Episódio")
axes[1, 1].set_ylabel("ε (epsilon)")
axes[1, 1].set_title("Decaimento do Epsilon")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RUN_DIR}/learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvo em {RUN_DIR}/learning_curves.png")

## 3. Mapa de Valores + Política Aprendida

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Heatmap de valores
plot_value_heatmap(env, agent, ax=axes[0])
axes[0].set_title("V(s) = max_a Q(s, a)")

# Valores + setas da política
plot_value_and_policy(env, agent, ax=axes[1])

# Caminho greedy
plot_greedy_path(env, agent, ax=axes[2])

plt.tight_layout()
plt.savefig(f"{RUN_DIR}/policy_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvo em {RUN_DIR}/policy_analysis.png")

## 4. Análise do Caminho Greedy

In [ ]:
# Executar o caminho greedy e medir
path = greedy_rollout(env, agent, max_steps=500)

last_state = path[-1]
last_rc = env.state_to_rc(last_state)
reached_goal = last_rc == env.goal

print(f"Caminho greedy: {len(path)} estados visitados")
print(f"Chegou ao objetivo? {'✅ SIM' if reached_goal else '❌ NÃO'}")
print(f"Posição final: {last_rc}")
print()

# Mostrar as primeiras e últimas posições
path_coords = [env.state_to_rc(s) for s in path]
print("Primeiros 10 passos:", path_coords[:10])
print("Últimos 5 passos:", path_coords[-5:])

## 5. Análise da Q-Table

In [ ]:
action_names = [
    "UP", "DOWN", "LEFT", "RIGHT", 
    "UP-LEFT", "UP-RIGHT", "DOWN-LEFT", "DOWN-RIGHT"
]

# Q-valores do estado inicial
start_state = env._rc_to_state(*env.start)
print("Q-valores no estado INICIAL (Start):")
for i, name in enumerate(action_names):
    print(f"  {name:>6s}: Q = {agent.Q[start_state, i]:>8.3f}")
print(f"  → Melhor ação: {action_names[agent.greedy_action(start_state)]}")
print()

# Estatísticas gerais da Q-table
free_mask = env.grid.flatten() == 0
Q_free = agent.Q[free_mask]  # apenas estados livres (sem obstáculos)

print("Estatísticas da Q-table (apenas estados livres):")
print(f"  Média:  {Q_free.mean():>8.3f}")
print(f"  Desvio: {Q_free.std():>8.3f}")
print(f"  Mín:    {Q_free.min():>8.3f}")
print(f"  Máx:    {Q_free.max():>8.3f}")

In [ ]:
# Distribuição dos Q-valores (apenas estados livres)
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(Q_free.flatten(), bins=80, color="#2196F3", alpha=0.7, edgecolor="white", linewidth=0.3)
ax.set_xlabel("Q(s, a)")
ax.set_ylabel("Frequência")
ax.set_title("Distribuição dos Q-valores (estados livres)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Análise por Fases do Treino

Dividimos o treino em 5 fases iguais para observar a evolução do aprendizado.

In [ ]:
n_phases = 5
phase_size = len(df) // n_phases

print(f"{'Fase':<8} {'Episódios':<18} {'Retorno Médio':>14} {'Compr. Médio':>14} {'Sucesso %':>10}")
print("-" * 68)

for i in range(n_phases):
    start_idx = i * phase_size
    end_idx = (i + 1) * phase_size if i < n_phases - 1 else len(df)
    phase = df.iloc[start_idx:end_idx]
    
    ep_range = f"{phase['episode'].iloc[0]}-{phase['episode'].iloc[-1]}"
    print(
        f"{i+1:<8} {ep_range:<18} "
        f"{phase['episode_return'].mean():>14.2f} "
        f"{phase['episode_length'].mean():>14.1f} "
        f"{phase['success'].mean() * 100:>9.1f}%"
    )